# 배치 목표를 알면 사람의 다음 조작·리듬을 더 잘 예측할까?
## 1. 왜 / 무엇
직전 행동만 보던 통계 기준선 다음으로, **관측 prefix를 읽는 작은 신경망**을 비교합니다.

- `prefix_only`: 관측한 조작·지난 간격만 사용합니다.
- `oracle_goal_prefix`: 같은 자료에 사람이 실제 선택한 배치 목표(조각 종류+셀)를 추가합니다.
- 각 비교군은 행동 MLP와 시간 MLP를 따로 학습합니다. 모두 **101 → ReLU 64 → softmax**이고 출력만 6개 행동/최대 16개 시간 지점으로 다릅니다.
- 첫 행동은 BOS에서 예측합니다. 첫 시간은 null이라 시간 loss에서 제외하고 실제 0ms는 그대로 사용합니다.
- 정답 목표와 Human 이력을 제공한 **one-step 진단**입니다. 배치 모델·자체 실행 AI·Hold 학습이 아닙니다.

[고정 명세](../../../ai-harness/ml-experiments/experiments/human-goal-conditioned-prefix.md) · [공식 MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)
**모두 실행 기본값은 가상 Smoke뿐입니다. 별도 Python 파일을 먼저 실행하지 않습니다.**


## 2. 손으로 읽는 작은 예제
`move_left@100 → rotate@100 → hard_drop@250` 중 rotate까지 관측했다면:

| 구분 | 값 |
|---|---|
| 공통 입력 | move_left → rotate, 지난 간격 [0ms] |
| 목표 없는 모델의 추가 입력 | goal 15차원 모두 0 |
| 목표 조건 모델의 추가 입력 | Human이 선택한 조각 종류 + 정렬된 네 셀 |
| 행동 정답 | hard_drop |
| 시간 정답 | 150ms |

현재 정답 hard_drop/150ms를 입력에 넣지 않습니다. 목표는 이 실험에만 허용한 oracle 조건입니다.
조각 종류도 조건에 포함되므로 결과는 **목표 묶음 전체 효과**이지 좌표만의 인과 효과가 아닙니다.


In [ ]:
import copy
import hashlib
import json
import os
import platform
import tempfile
import warnings
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_goal_conditioned_prefix.ipynb"
OFFLINE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_action_offline_views.ipynb"
EXPERIMENT_ID = "human-goal-conditioned-prefix"
OFFLINE_DEFS_SHA = "a3f694ff27fddc1c5dff0be9b2f6609230363bc9421329834fbe72e3293b7f5a"

def definition_sha(path):
    selected = [{"id": c["id"], "source": "".join(c["source"])}
                for c in json.loads(path.read_text(encoding="utf-8"))["cells"]
                if c["cell_type"] == "code" and "definitions" in c.get("metadata", {}).get("tags", [])]
    return hashlib.sha256((json.dumps(selected, ensure_ascii=False, sort_keys=True,
                                     allow_nan=False, indent=2) + "\n").encode("utf-8")).hexdigest()

assert definition_sha(OFFLINE_NOTEBOOK) == OFFLINE_DEFS_SHA
dependency = {"__name__": "goal_prefix_shared"}
shared_cell = next(c for c in json.loads(OFFLINE_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
                   if c["id"] == "shared-definitions")
exec(compile("".join(shared_cell["source"]), str(OFFLINE_NOTEBOOK), "exec"), dependency)
for name in ("require", "finite", "integer", "json_bytes", "sha256_file",
             "repo_input", "reject_constant", "finite_float", "physical_key"):
    globals()[name] = dependency[name]
checked_paths = dependency["link"]["checked_paths"]
publish_report = dependency["link"]["publish_report"]
dependency["link"]["dataset"]["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID

ACTIONS = ("hard_drop", "move_left", "move_right", "rotate", "rotate_ccw", "soft_drop")
TOKENS = ("<PAD>", "<BOS>", *ACTIONS)
PIECES = ("bar", "square", "tree", "ess", "zee", "eleft", "eright")
VARIANTS = ("prefix_only", "oracle_goal_prefix")
GOAL_SIZE, PREFIX_SIZE = 15, 86
FULL_EPOCHS, SMOKE_EPOCHS = 30, 3
EXPECTED_MANIFEST_SHA = "9ae27fc0b85903f412b434f5667e51376e7d5bb3170d3bab3d439f0e415f28bf"
EXPECTED_FILES = {
    "train": (6440, 6417, 47537, 41120, 149, "535e87540b5d9983b4065df2904df5702f13aa7ba40df640f6fcb7dfa332c08c"),
    "validation": (1519, 1516, 11691, 10175, 31, "e2d598e4792f523e32b599b152d2adde8dd53568c6ade91af4c78a1a604a303e"),
}


In [ ]:
def encode_input(prefix, goal):
    """명시적으로 전달받은 관측 prefix와 oracle 목표만 숫자로 바꾼다."""
    acts, intervals = prefix["prefixActions"], prefix["observedIntervalsMs"]
    require(isinstance(acts, list) and acts and all(isinstance(a, str) and a in TOKENS[1:] for a in acts),
            "PREFIX_ACTION_INVALID")
    require(("<BOS>" not in acts or acts == ["<BOS>"])
            and isinstance(intervals, list) and len(intervals) == len(acts) - 1
            and all(finite(t) and t >= 0 for t in intervals), "PREFIX_TIME_INVALID")
    recent = ["<PAD>"] * max(0, 8 - len(acts)) + acts[-8:]
    action_features = [float(a == token) for a in recent for token in TOKENS]
    counts = [float(np.log1p(acts.count(a))) for a in ACTIONS]
    recent_times = intervals[-8:]
    time_values = [0.0] * (8 - len(recent_times)) + [float(np.log1p(t)) for t in recent_times]
    time_mask = [0.0] * (8 - len(recent_times)) + [1.0] * len(recent_times)
    piece = goal["physicalClassKey"].split("|")[0]
    cells = sorted(goal["selectedCells"], key=lambda c: (c["row"], c["col"]))
    key = physical_key({"pieceType": piece, "rotation": 0, "row": 0, "col": 0, "cells": cells}, selected=True)
    require(key == goal["physicalClassKey"], "GOAL_CLASS_MISMATCH")
    goal_features = [float(piece == p) for p in PIECES] + [
        value for c in cells for value in (c["row"] / 19, c["col"] / 9)]
    vector = np.array(action_features + counts + time_values + time_mask + goal_features, dtype=float)
    require(vector.shape == (PREFIX_SIZE + GOAL_SIZE,) and np.isfinite(vector).all(), "FEATURE_INVALID")
    return vector

def flatten_views(segments, split):
    """두 비교에 사용할 공통 구간만 선택하고 입력과 정답을 분리한다."""
    rows, seen = [], set()
    for segment in segments:
        lineage = segment["lineage"]
        require(lineage["split"] == split and segment["support"]["policy"] == "offline-use-v1"
                and segment["placement"]["inferenceEligible"] is False, "VIEW_CONTRACT_INVALID")
        if not all(segment["eligibility"][k]["eligible"] for k in
                   ("placement_snapshot_eligible", "control_sequence_eligible", "oracle_goal_conditioned")):
            continue
        goal = segment["oracleCondition"]["humanGoal"]
        require(goal == segment["placement"]["target"]
                and segment["oracleCondition"]["diagnosticOnly"] is True, "ORACLE_LABEL_MISMATCH")
        for example in segment["control"]["examples"]:
            idx, target = example["currentInputIndex"], example["target"]
            require(integer(idx) and isinstance(target["action"], str) and target["action"] in ACTIONS,
                    "ACTION_TARGET_INVALID")
            value = target["intervalMs"]
            require((idx == 0 and value is None and example["input"]["prefixActions"] == ["<BOS>"])
                    or (idx > 0 and finite(value) and value >= 0), "TIME_TARGET_INVALID")
            identifier = segment["segmentId"] + ":" + str(idx).zfill(8)
            require(identifier not in seen, "DUPLICATE_EXAMPLE")
            seen.add(identifier)
            rows.append({"exampleId": identifier, "segmentId": segment["segmentId"],
                         "gameId": lineage["gameId"], "x": encode_input(example["input"], goal),
                         "action": ACTIONS.index(target["action"]), "time": value})
    rows.sort(key=lambda r: r["exampleId"])
    require(rows, "NO_COMMON_ELIGIBLE_EXAMPLES")
    return rows

def arrays(rows):
    x = np.stack([r["x"] for r in rows])
    y = np.array([r["action"] for r in rows], dtype=int)
    times = np.array([np.nan if r["time"] is None else r["time"] for r in rows])
    return x, y, times

def time_support(train_times):
    """Train만으로 0ms atom과 최대 15개 양의 시간 support를 정한다."""
    values = train_times[np.isfinite(train_times)]
    require(len(values) and (values >= 0).all() and np.any(values > 0), "TRAIN_TIME_SUPPORT_INVALID")
    return np.unique(np.r_[0.0, np.quantile(values[values > 0], np.linspace(0, 1, 15), method="nearest")])


In [ ]:
def load_views():
    """고정 Train/Validation만 읽는다. Test/raw/rejected 파일은 열지 않는다."""
    directory = repo_input(os.environ.get("HUMAN_GOAL_PREFIX_DATASET",
        "data/datasets/human-placement-action-offline-v1-20261102-run1"))
    paths = {s: directory / (s + ".jsonl") for s in EXPECTED_FILES}
    paths["manifest"] = directory / "manifest.json"
    before = {p: sha256_file(p) for p in paths.values()}
    require(before[paths["manifest"]] == EXPECTED_MANIFEST_SHA, "MANIFEST_SHA_MISMATCH")
    manifest = json.loads(paths["manifest"].read_text(encoding="utf-8"), parse_constant=reject_constant)
    require(manifest["status"] == "offline_dataset_constructed"
            and manifest["dataConstructionComplete"] is True
            and manifest["policy"] == "offline-use-v1"
            and manifest["modelFitCount"] == 0 and manifest["testPayloadRead"] is False,
            "DATASET_MANIFEST_INVALID")
    result = {}
    for split, expected in EXPECTED_FILES.items():
        require(before[paths[split]] == expected[-1]
                and manifest["files"][split + ".jsonl"]["sha256"] == expected[-1], "DATASET_SHA_MISMATCH")
        with paths[split].open("rb") as handle:
            segments = [json.loads(line, parse_constant=reject_constant, parse_float=finite_float) for line in handle]
        require(len(segments) == expected[0], "SEGMENT_COUNT_CHANGED")
        rows = flatten_views(segments, split)
        require((len({r["segmentId"] for r in rows}), len(rows),
                 sum(r["time"] is not None for r in rows), len({r["gameId"] for r in rows})) == expected[1:5],
                "COMMON_COHORT_CHANGED")
        result[split] = rows
    train_games = {r["gameId"] for r in result["train"]}
    validation_games = {r["gameId"] for r in result["validation"]}
    reserved = set(manifest["provenance"]["reservedTestGameIds"])
    require(len(reserved) == 33 and not train_games & validation_games
            and not (train_games | validation_games) & reserved, "GAME_SPLIT_LEAKAGE")
    require(all(sha256_file(p) == value for p, value in before.items()), "DATASET_CHANGED")
    return result, {"manifestSha256": EXPECTED_MANIFEST_SHA,
                    "datasetFiles": {s: EXPECTED_FILES[s][-1] for s in EXPECTED_FILES},
                    "reservedTestGameCount": 33, "testFileRead": False, "rawFileRead": False}, before


In [ ]:
def fit_head(x, y, classes, seed, epochs):
    """Validation을 받지 않고 고정 epoch만 학습한다. 예산 종료는 수렴 판정이 아니다."""
    model = MLPClassifier(hidden_layer_sizes=(64,), activation="relu", solver="adam",
                          alpha=0.0001, batch_size=min(128, len(y)), learning_rate_init=0.001,
                          shuffle=True, random_state=seed, early_stopping=False)
    losses = []
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        for epoch in range(epochs):
            model.partial_fit(x, y, classes=classes if epoch == 0 else None)
            require(np.isfinite(model.loss_), "NONFINITE_TRAIN_LOSS")
            losses.append(float(model.loss_))
    require(not caught, "UNEXPECTED_FIT_WARNING")
    require(all(np.isfinite(a).all() for a in (*model.coefs_, *model.intercepts_)), "NONFINITE_MODEL")
    return model, losses

def predict_probabilities(model, x, classes):
    require(np.array_equal(model.classes_, np.arange(classes)), "CLASS_ORDER_CHANGED")
    p = model.predict_proba(x)
    require(p.shape == (len(x), classes) and np.isfinite(p).all()
            and (p >= 0).all() and np.allclose(p.sum(axis=1), 1, atol=1e-12), "PROBABILITY_INVALID")
    return p

def train_variants(train, epochs):
    """동일한 Train-only 전처리와 예산에서 goal mask만 변경한다."""
    x, y, times = arrays(train)
    scaler = StandardScaler().fit(x)
    scaled = scaler.transform(x)
    support = time_support(times)
    mask = np.isfinite(times)
    time_y = np.argmin(np.abs(times[mask, None] - support[None, :]), axis=1)
    models = {}
    with threadpool_limits(limits=1):
        for variant in VARIANTS:
            features = scaled.copy()
            if variant == "prefix_only":
                features[:, -GOAL_SIZE:] = 0
            action, action_loss = fit_head(features, y, np.arange(6), 1729, epochs)
            timing, timing_loss = fit_head(features[mask], time_y, np.arange(len(support)), 1730, epochs)
            models[variant] = {"action": action, "timing": timing,
                               "trainLoss": {"action": action_loss, "timing": timing_loss}}
    return models, scaler, support, {
        "actionTrainExamples": len(y), "timingTrainExamples": int(mask.sum()),
        "scalerFitExamples": int(scaler.n_samples_seen_), "scalerMean": scaler.mean_.tolist(),
        "scalerScale": scaler.scale_.tolist(), "timeSupportMs": support.tolist(),
        "trainTimeQuantizationMaeMs": float(np.mean(abs(times[mask] - support[time_y]))),
        "epochs": epochs, "batchSizeAction": min(128, len(y)), "batchSizeTiming": min(128, int(mask.sum())),
        "optimizer": "adam", "learningRate": 0.001, "alpha": 0.0001, "seeds": [1729, 1730],
        "hiddenSizes": [64], "featureDimensions": PREFIX_SIZE + GOAL_SIZE,
        "actionOutputClasses": 6, "timingOutputClasses": len(support),
        "convergenceAssessed": False, "fitWarnings": 0,
        "weighting": "uniform_per_observed_example", "validationUsedForFit": False}


In [ ]:
def discrete_crps(p, support, observations):
    """분포별 CRPS를 원래 ms 정답으로 계산한다. 관측값을 support로 반올림하지 않는다."""
    before_p = np.cumsum(p, axis=1) - p
    before_pt = np.cumsum(p * support, axis=1) - p * support
    half_distance = np.sum(p * (support * before_p - before_pt), axis=1)
    result = np.sum(p * np.abs(support - observations[:, None]), axis=1) - half_distance
    require(np.isfinite(result).all() and (result >= -1e-8).all(), "CRPS_INVALID")
    return np.maximum(result, 0)

def evaluate_model(bundle, scaler, support, rows, variant):
    x, y, times = arrays(rows)
    features = scaler.transform(x)
    if variant == "prefix_only":
        features[:, -GOAL_SIZE:] = 0
    with threadpool_limits(limits=1):
        p = predict_probabilities(bundle["action"], features, 6)
        timing_mask = np.isfinite(times)
        pt = predict_probabilities(bundle["timing"], features[timing_mask], len(support))
    predicted = p.argmax(axis=1)
    assigned = p[np.arange(len(y)), y]
    nll = -np.log(np.maximum(assigned, 1e-15))
    crps_values = np.full(len(rows), np.nan)
    mae = np.full(len(rows), np.nan)
    brier = np.full(len(rows), np.nan)
    crps_values[timing_mask] = discrete_crps(pt, support, times[timing_mask])
    mae[timing_mask] = abs(np.sum(pt * support, axis=1) - times[timing_mask])
    brier[timing_mask] = (pt[:, 0] - (times[timing_mask] == 0)) ** 2
    games = np.array([r["gameId"] for r in rows])
    per_game = []
    for game in sorted(set(games)):
        idx = games == game
        ti = idx & timing_mask
        per_game.append({"gameId": game, "actionExamples": int(idx.sum()), "timingExamples": int(ti.sum()),
                         "actionNll": float(nll[idx].mean()),
                         "macroF1": float(f1_score(y[idx], predicted[idx], labels=np.arange(6),
                                                   average="macro", zero_division=0)),
                         "accuracy": float((y[idx] == predicted[idx]).mean()),
                         "crpsMs": float(crps_values[ti].mean()) if ti.any() else None,
                         "meanPredictionMaeMs": float(mae[ti].mean()) if ti.any() else None,
                         "zeroMsBrier": float(brier[ti].mean()) if ti.any() else None})
    metric_names = ("actionNll", "macroF1", "accuracy", "crpsMs", "meanPredictionMaeMs", "zeroMsBrier")
    macro = {k: float(np.mean([g[k] for g in per_game if g[k] is not None])) for k in metric_names}
    precision, recall, f1, counts = precision_recall_fscore_support(y, predicted, labels=np.arange(6), zero_division=0)
    return {"gameMacro": macro, "perGame": per_game,
            "gameMacroDenominators": {"actionGames": len(per_game),
                                     "timingGames": sum(g["timingExamples"] > 0 for g in per_game)},
            "perAction": [{"action": a, "trueSupport": int(counts[i]),
                           "predictedSupport": int((predicted == i).sum()),
                           "precision": float(precision[i]), "recall": float(recall[i]), "f1": float(f1[i])}
                          for i, a in enumerate(ACTIONS)],
            "confusionMatrix": confusion_matrix(y, predicted, labels=np.arange(6)).tolist(),
            "probabilityLogClipCount": int((assigned < 1e-15).sum()),
            "meanZeroMsProbability": float(pt[:, 0].mean()),
            "observedZeroMsRate": float((times[timing_mask] == 0).mean()),
            "actionExamples": len(y), "timingExamples": int(timing_mask.sum())}

def run_experiment(data, scope, provenance, epochs):
    require(scope in ("synthetic", "full")
            and epochs == (SMOKE_EPOCHS if scope == "synthetic" else FULL_EPOCHS), "BUDGET_OR_SCOPE_INVALID")
    require((platform.python_version(), np.__version__, sklearn.__version__)
            == ("3.12.13", "2.5.1", "1.9.0"), "PINNED_TRAINING_ENVIRONMENT_REQUIRED")
    require(not {r["gameId"] for r in data["train"]} & {r["gameId"] for r in data["validation"]},
            "GAME_SPLIT_LEAKAGE")
    models, scaler, support, training = train_variants(data["train"], epochs)
    results = {v: evaluate_model(models[v], scaler, support, data["validation"], v) for v in VARIANTS}
    for head in ("action", "timing"):
        assert [a.shape for a in models[VARIANTS[0]][head].coefs_] == [
            a.shape for a in models[VARIANTS[1]][head].coefs_]
    a, b = (results[v]["gameMacro"] for v in VARIANTS)
    delta = {k: b[k] - a[k] for k in a}
    signal = ("oracle_goal_signal_observed" if delta["actionNll"] < -1e-12
              and delta["macroF1"] >= -1e-12 and delta["crpsMs"] <= 1e-9 else
              "goal_tradeoff_requires_review" if delta["actionNll"] < -1e-12 else "no_clear_goal_gain")
    per_game_delta = [{"gameId": a["gameId"], **{k: b[k] - a[k] for k in delta
                       if a[k] is not None and b[k] is not None}}
                      for a, b in zip(results[VARIANTS[0]]["perGame"], results[VARIANTS[1]]["perGame"])]
    return {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID,
            "runMode": "full" if scope == "full" else "smoke", "scope": scope,
            "fullCohortCovered": scope == "full", "policyDecisionEligible": scope == "full",
            "canonicalResultEligible": False, "productionIntegration": False,
            "wholePolicyEvidence": False, "oracleGoalProvided": True, "humanPrefixProvided": True,
            "boardSnapshotUsed": False, "holdSupported": False, "firstWaitSupported": False,
            "modelFitCount": 4, "analysisStatus": signal if scope == "full" else "smoke_only_no_policy_decision",
            "decisionScope": "offline_oracle_condition_signal_only", "training": training,
            "results": results, "oracleMinusPrefixGameMacro": delta, "perGameDelta": per_game_delta,
            "trainingLoss": {v: models[v]["trainLoss"] for v in VARIANTS},
            "sourceDefinitionSha256": definition_sha(SOURCE_NOTEBOOK),
            "offlineDefinitionSha256": OFFLINE_DEFS_SHA,
            "environment": {"python": platform.python_version(), "numpy": np.__version__,
                            "sklearn": sklearn.__version__, "blasThreads": 1},
            "provenance": provenance}

def save_report(report, output):
    require(isinstance(output, str) and bool(output), "NEW_REPORT_PATH_REQUIRED")
    path = Path(output)
    if not path.is_absolute():
        path = ROOT / path
    require(path.parent.is_dir(), "REPORT_PARENT_MISSING")
    destination = checked_paths(ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(path), report["runMode"])[1]
    publish_report(report, destination)


In [ ]:
def synthetic_data():
    base, refs, _ = dependency["link"]["synthetic_case"]()
    raw = base[1]["payload"]
    goal = {"physicalClassKey": physical_key(raw["selectedPlacement"], selected=True),
            "selectedCells": raw["selectedPlacement"]["cells"]}
    data = {}
    for split, games in (("train", 3), ("validation", 2)):
        rows = []
        for game in range(games):
            for piece in range(2):
                piece_goal = copy.deepcopy(goal)
                if piece == 1:
                    for cell in piece_goal["selectedCells"]:
                        cell["col"] += 6
                    piece_goal["physicalClassKey"] = physical_key(
                        {"pieceType": "square", "rotation": 0, "row": 18, "col": 6,
                         "cells": piece_goal["selectedCells"]}, selected=True)
                actions = ("move_left", "rotate", "hard_drop") if piece == 0 else ("move_right", "move_left", "hard_drop")
                for i in range(3):
                    prefix = {"prefixActions": ["<BOS>"] if i == 0 else list(actions[:i]),
                              "observedIntervalsMs": [] if i < 2 else [0]}
                    rows.append({"exampleId": f"{split}-{game}-{piece}-{i}", "segmentId": f"{split}-{game}-{piece}",
                                 "gameId": f"{split}-{game}", "x": encode_input(prefix, piece_goal),
                                 "action": ACTIONS.index(actions[i]), "time": None if i == 0 else 0 if i == 1 else 150})
        data[split] = rows
    return data

def self_checks():
    data = synthetic_data()
    x, y, t = arrays(data["train"])
    assert x.shape == (18, 101) and np.isnan(t).sum() == 6 and (t == 0).sum() == 6
    scaler = StandardScaler().fit(x)
    changed_validation = copy.deepcopy(data["validation"])
    for row in changed_validation:
        row["x"] = row["x"] + 999
    scaler.transform(arrays(changed_validation)[0])
    assert scaler.n_samples_seen_ == 18 and np.allclose(scaler.mean_, x.mean(axis=0))
    support = time_support(t)
    assert support.tolist() == [0.0, 150.0]
    probabilities = np.array([[0.25, 0.75], [0.5, 0.5]])
    observed = np.array([0.0, 75.0])
    brute = [sum(p * abs(support - actual)) -
             0.5 * sum(p[i] * p[j] * abs(a - b) for i, a in enumerate(support) for j, b in enumerate(support))
             for p, actual in zip(probabilities, observed)]
    assert np.allclose(discrete_crps(probabilities, support, observed), brute)
    masked = scaler.transform(x)
    masked[:, -GOAL_SIZE:] = 0
    assert not masked[:, -GOAL_SIZE:].any()
    assert np.array_equal(masked[:, :PREFIX_SIZE], scaler.transform(x)[:, :PREFIX_SIZE])
    assert np.any(scaler.transform(x)[:, -GOAL_SIZE:] != 0)
    raw = dependency["link"]["synthetic_case"]()[0][1]["payload"]["selectedPlacement"]
    target = {"physicalClassKey": physical_key(raw, selected=True), "selectedCells": raw["cells"]}
    prefix = {"prefixActions": ["move_left"], "observedIntervalsMs": []}
    encoded = encode_input(prefix, target)
    reverse_target = {**target, "selectedCells": list(reversed(target["selectedCells"]))}
    assert np.array_equal(encoded, encode_input(prefix, reverse_target))
    assert np.array_equal(encoded, encode_input({**prefix, "nextAction": "hard_drop",
                                                 "futureTime": 9999, "outcome": "win"}, target))
    for bad in ({"prefixActions": ["hold"], "observedIntervalsMs": []},
                {"prefixActions": ["move_left"], "observedIntervalsMs": [0]}):
        raw = dependency["link"]["synthetic_case"]()[0][1]["payload"]["selectedPlacement"]
        goal = {"physicalClassKey": physical_key(raw, selected=True), "selectedCells": raw["cells"]}
        try:
            encode_input(bad, goal)
        except ValueError:
            pass
        else:
            raise AssertionError("INVALID_PREFIX_ACCEPTED")
    with tempfile.TemporaryDirectory() as directory:
        p = Path(directory).resolve() / "report.json"
        fake = {"runMode": "smoke", "synthetic": True}
        save_report(fake, str(p))
        try:
            save_report(fake, str(p))
        except ValueError:
            pass
        else:
            raise AssertionError("REPORT_OVERWRITE_ALLOWED")
        (p.parent / "alias.json").symlink_to(p)
        (p.parent / "dangling.json").symlink_to(p.parent / "absent.json")
        os.link(p, p.parent / "hardlink.json")
        for name in ("alias.json", "dangling.json", "hardlink.json"):
            try:
                save_report(fake, str(p.parent / name))
            except ValueError:
                pass
            else:
                raise AssertionError("OUTPUT_ALIAS_ACCEPTED")
    return "PASS: BOS/null/0ms, feature shape, goal mask, invalid prefix, exact CRPS, exclusive report"


In [ ]:
def show_report(report):
    print(report["scope"], "|", report["analysisStatus"],
          "| oracle + Human prefix diagnosis, NOT a complete gameplay AI")
    display(pd.DataFrame({v: report["results"][v]["gameMacro"] for v in VARIANTS}).T)
    display(pd.DataFrame([report["training"]]).drop(
        columns=["scalerMean", "scalerScale", "timeSupportMs"]))
    display(pd.DataFrame(report["perGameDelta"]))
    per_action = pd.DataFrame([{**r, "variant": v} for v in VARIANTS for r in report["results"][v]["perAction"]])
    display(per_action)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for v in VARIANTS:
        for head in ("action", "timing"):
            axes[0].plot(report["trainingLoss"][v][head], label=v + "/" + head)
    axes[0].set(xlabel="epoch (fixed budget)", ylabel="Train CE + L2", title="Optimization, not held-out performance")
    axes[0].legend(fontsize=6)
    per_action.pivot(index="action", columns="variant", values="recall").plot.bar(ax=axes[1], rot=30)
    axes[1].set(ylabel="Validation recall", title="Per-action support matters", ylim=(0, 1))
    deltas = pd.DataFrame(report["perGameDelta"])
    axes[2].bar(range(len(deltas)), deltas["actionNll"])
    axes[2].axhline(0, color="black", linewidth=1)
    axes[2].set(xlabel="Validation game (sorted)", ylabel="oracle - prefix NLL",
                title="Below 0 favors oracle goal")
    plt.tight_layout(); plt.show(); plt.close(fig)
    print("Time support (ms):", report["training"]["timeSupportMs"])
    print("No Validation tuning / no Test / no placement fit / no autonomous rollout.")


## 3. 가상 Smoke — 설정
Train 3게임/18 행동 정답, Validation 2게임/12 행동 정답으로 **동일 학습·평가 함수**를 3 epoch 점검합니다.
학습 효과를 판정할 만큼의 자료가 아닙니다. 실제 dataset은 열지 않습니다.


In [ ]:
SMOKE_REPORT = os.environ.get("HUMAN_GOAL_PREFIX_SMOKE_REPORT") or None


In [ ]:
print(self_checks())
smoke_data = synthetic_data()
smoke_report = run_experiment(smoke_data, "synthetic", {"syntheticOnly": True, "testFileRead": False}, SMOKE_EPOCHS)
if SMOKE_REPORT:
    save_report(smoke_report, SMOKE_REPORT)


In [ ]:
show_report(smoke_report)


## 4. Full — 별도 설정
이전 오프라인 dataset의 **공통 적격 표본**만 사용합니다. 실제 Full은 사용자가 실행합니다.

1. 위 Smoke를 확인하고 `RUN_FULL=True`로 바꿉니다.
2. `FULL_REPORT`에 **새 JSON 경로**를 입력합니다. 기본 제안은 아래에 있고 기존 파일은 덮어쓰지 않습니다.
3. 부모 폴더가 없다면 아래 설정 셀에서 실험 전용 폴더만 생성합니다.
4. 실행 셀 뒤의 결과 셀에서 학습 loss·Validation 행동별 표·게임별 차이를 확인합니다.

30 epoch 고정이며 Validation 결과로 모델을 다시 고르지 않습니다. sklearn CPU 학습이고 GPU는 필요 조건이 아닙니다.
전체 재현 시 CPU/BLAS·버전 차이로 마지막 소수점 차이가 있을 수 있습니다. 모델 artifact는 저장하지 않습니다.


In [ ]:
RUN_FULL = False
FULL_REPORT = "data/ml-experiments/human-goal-conditioned-prefix/full-report-run1.json"
# Example: "data/ml-experiments/human-goal-conditioned-prefix/full-report-run1.json"


In [ ]:
full_report = None
if RUN_FULL:
    require(FULL_REPORT is not None, "SET_NEW_FULL_REPORT_PATH")
    report_path = Path(FULL_REPORT)
    if not report_path.is_absolute():
        report_path = ROOT / report_path
    allowed_parent = ROOT / "data/ml-experiments" / EXPERIMENT_ID
    require(".." not in report_path.parts and not any(
        p.is_symlink() for p in (report_path, *report_path.parents)), "OUTPUT_ALIAS_FORBIDDEN")
    require(report_path.parent == allowed_parent or report_path.parent == Path("/private/tmp")
            or str(report_path.parent).startswith("/private/tmp/"),
            "USE_EXPERIMENT_REPORT_DIRECTORY")
    report_path.parent.mkdir(parents=True, exist_ok=True)
    checked_paths(ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(report_path), "full")
    source_defs_before = definition_sha(SOURCE_NOTEBOOK)
    data, provenance, watched = load_views()
    full_report = run_experiment(data, "full", provenance, FULL_EPOCHS)
    require(all(sha256_file(p) == h for p, h in watched.items())
            and definition_sha(SOURCE_NOTEBOOK) == source_defs_before
            and definition_sha(OFFLINE_NOTEBOOK) == OFFLINE_DEFS_SHA, "INPUT_OR_CODE_CHANGED")
    full_report["sourceUnchanged"] = True
    save_report(full_report, str(report_path))
    print("Saved:", str(report_path), "| SHA-256:", sha256_file(report_path))
else:
    print("Full skipped. Set RUN_FULL=True and a fresh FULL_REPORT path after approval.")


In [ ]:
if full_report is not None:
    show_report(full_report)


## 5. 결과 해석 / 한계 / 다음 gate
- 행동 NLL·시간 CRPS는 낮을수록, F1/recall은 높을수록 좋습니다. 차이는 oracle−prefix입니다.
- 이전 joint/B3와는 공통 구간·BOS 포함 여부·모델 입력이 다릅니다. 예전 점수와 직접 비교하지 않고 이번 두 비교군을 같은 표본에서 비교합니다.
- 학습 loss 감소는 Train에 대한 적합입니다. Validation 개선·수렴·사람다움의 증명과는 다릅니다.
- 정답 목표를 추가한 개선은 목표가 도움이 될 가능성의 근거입니다. AI가 스스로 목표를 선택한다는 증거가 아닙니다.
- 시간은 Train에서 정한 최대 16점의 분포입니다. 원래 ms로 채점하므로 양자화와 긴 tail 표현 한계도 점수에 반영됩니다.
- 실제 0ms를 보존하고 첫 대기시간을 0으로 채우지 않습니다. Hold·행동별 보드·공격에 대한 전략적 반응은 미지원입니다.
- 최근 8개 행동과 누적 횟수를 읽는 작은 MLP입니다. 전체 순서를 기억하는 GRU/Transformer나 공유 encoder를 채택한 것이 아닙니다.
- 다음은 사용자 Full 결과 해석입니다. Human 배치 모델·예측 목표 rollout·통합형 B·runtime을 자동 시작하지 않습니다.

